Environment: the `etunc` kernel (pixi env `etunc`, registered with `pixi run kernel`).

# Bin edges pooled across all datasets

`scripts/bin_obs.py`, `bin_cmip.py`, `bin_fppe.py` and `bin_lens.py` each build their own area mask and their own LAI/AI quantile edges. Comparing the datasets bin by bin needs one area mask and one set of edges for all of them. This notebook:

1. loads the binning inputs (climatological LAI and AI) of each dataset with its script's own settings and loaders, on the common `cfg.TARGET_RES` grid (cached, since loading takes a while, mostly for CMIP6);
2. builds one common area mask: land gridcells where ET (any year), LAI and AI are valid in every member of every dataset;
3. checks the hypothesis that CMIP6 spans the widest range of LAI and AI;
4. builds quantile edges pooled across all datasets and compares them with each dataset's own pooled edges, both on the common area;
5. saves the area mask and the pooled edges.

The "members" a dataset pools over, as in its script:

| dataset | members | edges in the script |
|---|---|---|
| obs | (ET, LAI, pr, rn) product combinations | `pooled_obs` |
| cmip | models (member mean of each model) | `pooled_cmip` |
| fppe | FHIST PPE parameter members | `pooled_fppe` |
| lens | CESM2-LE members | `pooled_lens` |

The periods differ: obs uses the years shared by each combination, the models the `TIME_SLICE` of their script.

In [ ]:
from __future__ import annotations

import itertools
import sys
from pathlib import Path

import numpy as np
import pandas as pd
import xarray as xr
import matplotlib.pyplot as plt

import etunc
import etunc.config as cfg
import etunc.temporal as temporal
import etunc.binning as binning
import etunc.plotting as plotting
import etunc.grid as rg
import etunc.load.cesm as lc
import etunc.load.cmip as cmip
import etunc.load.era5 as le
import etunc.load.ilamb as il
import etunc.load.obs as lo
from etunc.load.cmip import CMIPESGFLoader

# The driver scripts, for their settings and loaders (main() is not run on import)
sys.path.insert(0, str(Path(etunc.__file__).resolve().parents[1] / "scripts"))
import bin_obs, bin_cmip, bin_fppe, bin_lens

# The scripts call matplotlib.use("Agg") on import; restore inline figures
%matplotlib inline

In [2]:
# Dataset name -> driver script
SCRIPTS = {"obs": bin_obs, "cmip": bin_cmip, "fppe": bin_fppe, "lens": bin_lens}
LABELS = {"obs": "obs", "cmip": "CMIP6", "fppe": "FHIST PPE", "lens": "CESM2-LE"}
COLORS = {"obs": "k", "cmip": "tab:blue", "fppe": "tab:orange", "lens": "tab:green"}

N_BINS = {"lai": cfg.N_YBINS, "ai": cfg.N_XBINS}
TARGET_RES = cfg.TARGET_RES
GRID_TAG = rg.grid_tag(TARGET_RES)

# Weighting of the edges pooled across all datasets:
#   "dataset": each dataset weighs equally (each member equally within a dataset)
#   "value"  : every finite value counts once (binning.pooled_bin_edges), so datasets
#              with more members weigh more (50 LENS + 27 FPPE members are all CESM2)
WEIGHTING = "dataset"

REBUILD = False  # reload the binning inputs even if cached (set after changing a script's settings)
SAVE = True      # write the cached inputs, area mask, edges and figures

CACHE_ROOT = cfg.PROC_ROOT / "all" / "inputs"
PROC_OUT = cfg.PROC_ROOT / "all"
EDGES_OUT = cfg.BIN_EDGES_ROOT
FIG_OUT = cfg.FIG_ROOT / "all" / "qbin_edges"

# Every script must put its data on the same grid
assert all(m.TARGET_RES == TARGET_RES for m in SCRIPTS.values()), {k: m.TARGET_RES for k, m in SCRIPTS.items()}
mask = rg.land_mask(rg.target_grid(TARGET_RES)).sel(lat=cfg.LAT_BNDS)

In [3]:
def save_fig(fig, name: str):
    """Save `fig` to FIG_OUT (if SAVE) and keep it open for display."""
    if SAVE:
        fout = FIG_OUT / name
        fout.parent.mkdir(parents=True, exist_ok=True)
        fig.savefig(fout, dpi=cfg.DPI, bbox_inches="tight")
        print(fout)

## 1. Binning inputs of each dataset

Each loader repeats its script's `main()` up to `binning.prepare_inputs`, with the script's settings and the `load_annual` of the matching load module. It returns `lai` and `ai` (member, lat, lon), and `valid` (lat, lon): the land gridcells where ET (any year), LAI and AI are valid in every member (`binning.valid_area`), i.e. the script's own area mask.

In [4]:
def as_dataset(lai: xr.DataArray, ai: xr.DataArray, valid: xr.DataArray, members: list[str], period: str) -> xr.Dataset:
    """What one dataset contributes to the pooled edges, with `member` labelled by `members`."""
    fields = {
        "lai": lai.reset_coords(drop=True).assign_coords(member=members),
        "ai": ai.reset_coords(drop=True).assign_coords(member=members),
        "valid": valid.reset_coords(drop=True),
    }
    return xr.Dataset(fields, attrs={"time_period": period})


def concat_members(das: list[xr.DataArray]) -> xr.DataArray:
    return xr.concat(das, dim="member", coords="minimal", compat="override", join="exact", combine_attrs="drop_conflicts")


def cached(name: str, loader) -> xr.Dataset:
    """Binning inputs of dataset `name` from the cache, or from `loader()` (then cached if SAVE)."""
    fpath = CACHE_ROOT / f"{name}.lai_ai_clim.{GRID_TAG}.nc"
    if fpath.exists() and not REBUILD:
        print(f"{name}: reading {fpath}")
        return xr.load_dataset(fpath)
    print(f"=== {name}: loading ===")
    ds = loader()
    if SAVE:
        fpath.parent.mkdir(parents=True, exist_ok=True)
        ds.to_netcdf(fpath)
        print(fpath)
    return ds

In [ ]:
def load_obs() -> xr.Dataset:
    """bin_obs.main(): every product, then every (et, lai, pr, rn) combination with >= MIN_YEARS shared years."""
    m = bin_obs
    ann = {
        v: {p: il.load_annual(v, p, slice(None, None), TARGET_RES)[v] for p in (m.ILAMB_PRODUCTS.get(v) or products)}
        for v, products in il.PRODUCTS.items()
    }
    if m.ERA5_VARIABLES:
        for v, da in le.load_annual(m.ERA5_VARIABLES, m.era5_span(ann), TARGET_RES).items():
            ann[v]["ERA5"] = da
    if m.GRIDDED_ET_PRODUCTS:
        years = set().union(*(
            temporal.shared_years(*das) for das in itertools.product(*(ann[v].values() for v in m.FACTORS[1:]))
        ))
        span = slice(str(min(years)), str(max(years)))
        for label, (dataset, version, var, freq) in m.GRIDDED_ET_PRODUCTS.items():
            et = lo.load_annual(dataset, var, span, TARGET_RES, version=version, freq=freq)[var]
            ann["et"][label] = et.rename("et")
    ann["et"] = {p: m.drop_et_outliers(da, p) for p, da in ann["et"].items()}

    inputs, all_years = {}, []
    for combo in itertools.product(*(ann[v] for v in m.FACTORS)):
        years = temporal.shared_years(*(ann[v][p] for v, p in zip(m.FACTORS, combo)))
        if len(years) >= m.MIN_YEARS:
            inputs["-".join(combo)] = m.combo_inputs(ann, combo, years, mask)
            all_years += years
    print(f"{len(inputs)} combinations with >= {m.MIN_YEARS} shared years")

    valid = mask == 1
    for inp in inputs.values():
        valid = valid & binning.valid_area(inp)
    return as_dataset(
        concat_members([inp["lai"] for inp in inputs.values()]),
        concat_members([inp["ai"] for inp in inputs.values()]),
        valid, list(inputs), temporal.period_str(all_years),
    )


def load_cmip() -> xr.Dataset:
    """bin_cmip.main(): every model with all VARIABLES and sftlf; one member = the member mean of a model."""
    m = bin_cmip
    loader = CMIPESGFLoader(m.CATALOG)
    avail = cmip.available_members(loader.catalog, m.VARIABLES, m.EXPERIMENT_ID)
    sftlf = cmip.sftlf_files(m.FX_CATALOG)
    lai, ai, sids, valid = [], [], [], mask == 1
    for sid in sorted(set(m.SOURCE_IDS or avail) - set(m.OMIT_SOURCE_IDS)):
        if sid not in avail or sid not in sftlf:
            print(f"{sid}: not all of {m.VARIABLES} or no sftlf, skipping")
            continue
        mids = cmip.select_members(sid, avail[sid], m.MEMBER_IDS, m.DEFAULT_MEMBERS)
        if not mids:
            continue
        try:
            ann = cmip.load_annual(
                loader, sid, mids, sftlf[sid], m.TIME_SLICE, TARGET_RES, variables=m.VARIABLES,
                experiment_id=m.EXPERIMENT_ID, mask=mask, na_thres=cfg.NA_THRESH,
            )
        except ValueError as err:  # e.g. sftlf on a different grid than the data
            print(f"{sid}: {err}, skipping")
            continue
        inp = binning.prepare_inputs(et=ann["et"], lai=ann["lai"], precip=ann["pr"], rn=ann["rn"], mask=mask)
        valid = valid & binning.valid_area(inp, member_dim="member")
        # As for "pooled_cmip": each model contributes its member mean once
        lai.append(inp["lai"].mean("member", keep_attrs=True))
        ai.append(inp["ai"].mean("member", keep_attrs=True))
        sids.append(sid)
    return as_dataset(concat_members(lai), concat_members(ai), valid, sids, temporal.format_time_period(m.TIME_SLICE))


def load_cesm(source: str, m) -> xr.Dataset:
    """bin_fppe.main() / bin_lens.main(): every member of the CESM ensemble `source`, labelled by member_id."""
    ann = lc.load_annual(
        source, m.MEMBERS, m.TIME_SLICE, TARGET_RES, variables=m.VARIABLES, frequency=m.FREQUENCY,
        stream=m.STREAM, mask=mask, na_thres=cfg.NA_THRESH,
        **({"drop_outliers": m.DROP_OUTLIERS} if source == "fppe" else {"bb": m.BB}),
    )
    inp = binning.prepare_inputs(et=ann["et"], lai=ann["lai"], precip=ann["pr"], rn=ann["rn"], mask=mask)
    names = inp["lai"]["member_id"].values.tolist()
    valid = (mask == 1) & binning.valid_area(inp, member_dim="member")
    return as_dataset(inp["lai"], inp["ai"], valid, names, temporal.format_time_period(m.TIME_SLICE))


LOADERS = {
    "obs": load_obs,
    "cmip": load_cmip,
    "fppe": lambda: load_cesm("fppe", bin_fppe),  # members labelled by parameter name
    "lens": lambda: load_cesm("lens", bin_lens),  # members labelled by ens_name
}

In [6]:
inputs = {name: cached(name, loader) for name, loader in LOADERS.items()}
for name, ds in inputs.items():
    print(f"{name:5}: {ds.sizes['member']:3} members, {ds.attrs['time_period']}, "
          f"{int(ds.valid.sum())} valid gridcells (own area mask)")

=== obs: loading ===
et/CLASS: regridding 360x720 -> 1 deg
et  CLASS       : ('year', 'lat', 'lon') (7, 148, 360) 200301-200912 [-2.59, 126] W/m2
et/DOLCE: regridding 360x720 -> 1 deg
et  DOLCE       : ('year', 'lat', 'lon') (10, 148, 360) 200001-200912 [1.03, 105] W/m2
et/FLUXCOM: regridding 360x720 -> 1 deg
et  FLUXCOM     : ('year', 'lat', 'lon') (37, 148, 360) 198001-201612 [-0.818, 168] W/m2
et/GLEAMv3.3a: regridding 360x720 -> 1 deg
et  GLEAMv3.3a  : ('year', 'lat', 'lon') (39, 148, 360) 198001-201812 [-2.2, 154] W/m2
et/MOD16A2: regridding 360x720 -> 1 deg
et  MOD16A2     : ('year', 'lat', 'lon') (15, 148, 360) 200001-201412 [0.295, 130] W/m2
et/MODIS: regridding 360x720 -> 1 deg
et  MODIS       : ('year', 'lat', 'lon') (14, 148, 360) 200001-201312 [0.293, 130] W/m2
et  WECANN      : ('year', 'lat', 'lon') (9, 148, 360) 200701-201512 [-0.732, 104] W/m2


/glade/work/bbuchovecky/pixi/envs/etunc-9438097516130772666/envs/default/lib/python3.12/site-packages/xarray/conventions.py:204: SerializationWarning: variable 'lai' has multiple fill values {np.float32(1e+36), np.float32(-999.0)} defined, decoding all values to NaN.
  var = coder.decode(var, name=name)


lai/MODIS: regridding 360x720 -> 1 deg
lai MODIS       : ('year', 'lat', 'lon') (6, 148, 360) 200001-200512 [0, 5.23] m2/m2
pr/GPCPv2.3: regridding 360x720 -> 1 deg
pr  GPCPv2.3    : ('year', 'lat', 'lon') (40, 148, 360) 197901-201812 [0.01, 548] W/m2
rns CERESed4.2  : ('year', 'lat', 'lon') (22, 148, 360) 200101-202212 [-21.9, 220] W/m2
et  ERA5        : ('year', 'lat', 'lon') (6, 148, 360) 200001-200512 [-2, 124] W/m2
pr  ERA5        : ('year', 'lat', 'lon') (6, 148, 360) 200001-200512 [4.84e-05, 980] W/m2
rns ERA5        : ('year', 'lat', 'lon') (6, 148, 360) 200001-200512 [-37.4, 162] W/m2
pml V2.2a-MODIS monthly ET: regridded 6 files to 1 deg (load 3.1s, regrid 78.8s)
et  PMLv2.2a_MODIS: ('year', 'lat', 'lon') (5, 148, 360) 200101-200512 [0, 125] W/m2
pml V2.2c monthly ET: regridded 6 files to 1 deg (load 3.1s, regrid 10.0s)
et  PMLv2.2c    : ('year', 'lat', 'lon') (6, 148, 360) 200001-200512 [0, 125] W/m2
gleam v4.3a monthly E: regridded 6 files to 1 deg (load 23.5s, regrid 119.4

/glade/u/home/bbuchovecky/projects/et_unc/etunc/load/obs.py:395: UserWarning: gleam v4.3b monthly E (native): no files for years [2000, 2001, 2002]
  warnings.warn(f"{spec.name} {version} {freq} {var} ({res}): no files for years {missing}")


gleam v4.3b monthly E: regridded 3 files to 1 deg (load 28.3s, regrid 21.3s)
et/GLEAMv4.3b: 311 gridcell-years with annual ET < -1.0 W/m2 set to NaN (min -172)
et  GLEAMv4.3b  : ('year', 'lat', 'lon') (3, 148, 360) 200301-200512 [-0.0287, 149] W/m2
sith v2 monthly ET: regridded 1 files to 1 deg (load 6.4s, regrid 2.9s)
et  SiTHv2      : ('year', 'lat', 'lon') (6, 148, 360) 200001-200512 [0.294, 142] W/m2
48 combinations with >= 3 shared years
/glade/work/bbuchovecky/et_unc/proc/all/inputs/obs.lai_ai_clim.1deg.nc
=== cmip: loading ===
ACCESS-ESM1-5   : 1 member(s), native 145x192 -> ('member', 'year', 'lat', 'lon') (1, 20, 148, 360), years 199501-201412
AWI-ESM-1-1-LR  : 1 member(s), native 96x192 -> ('member', 'year', 'lat', 'lon') (1, 20, 148, 360), years 199501-201412
AWI-ESM-1-REcoM: not all of ['evspsbl', 'lai', 'pr', 'rsds', 'rsus', 'rlds', 'rlus'] or no sftlf, skipping
BCC-CSM2-MR: not all of ['evspsbl', 'lai', 'pr', 'rsds', 'rsus', 'rlds', 'rlus'] or no sftlf, skipping
BCC-ESM1 

...

/glade/work/bbuchovecky/pixi/envs/etunc-9438097516130772666/envs/default/lib/python3.12/site-packages/xarray/conventions.py:204: SerializationWarning: variable 'sftlf' has multiple fill values {np.float32(1e+20), np.float64(1e+20)} defined, decoding all values to NaN.
  var = coder.decode(var, name=name)


CESM2-WACCM     : 1 member(s), native 192x288 -> ('member', 'year', 'lat', 'lon') (1, 20, 148, 360), years 199501-201412


/glade/work/bbuchovecky/pixi/envs/etunc-9438097516130772666/envs/default/lib/python3.12/site-packages/xarray/conventions.py:204: SerializationWarning: variable 'sftlf' has multiple fill values {np.float32(1e+20), np.float64(1e+20)} defined, decoding all values to NaN.
  var = coder.decode(var, name=name)


CESM2-WACCM-FV2 : 1 member(s), native 96x144 -> ('member', 'year', 'lat', 'lon') (1, 20, 148, 360), years 199501-201412
CMCC-CM2-HR4: not all of ['evspsbl', 'lai', 'pr', 'rsds', 'rsus', 'rlds', 'rlus'] or no sftlf, skipping
CMCC-CM2-SR5    : 1 member(s), native 192x288 -> ('member', 'year', 'lat', 'lon') (1, 20, 148, 360), years 199501-201412
CMCC-ESM2       : 1 member(s), native 192x288 -> ('member', 'year', 'lat', 'lon') (1, 20, 148, 360), years 199501-201412
CNRM-CM6-1      : 1 member(s), native 128x256 -> ('member', 'year', 'lat', 'lon') (1, 20, 148, 360), years 199501-201412
CNRM-CM6-1-HR   : 1 member(s), native 360x720 -> ('member', 'year', 'lat', 'lon') (1, 20, 148, 360), years 199501-201412
CNRM-ESM2-1     : 1 member(s), native 128x256 -> ('member', 'year', 'lat', 'lon') (1, 20, 148, 360), years 199501-201412
CanESM5         : 1 member(s), native 64x128 -> ('member', 'year', 'lat', 'lon') (1, 20, 148, 360), years 199501-201412
CanESM5-1       : 1 member(s), native 64x128 -> ('m

KeyboardInterrupt: 

## 2. Common area mask

Land gridcells valid in every member of every dataset: the intersection of the four scripts' own area masks.

In [ ]:
valid = xr.concat([ds.valid for ds in inputs.values()], dim="product").assign_coords(product=list(inputs))
area = valid.all("product").rename("area_mask")
for name in inputs:
    print(f"{name:5}: {int(valid.sel(product=name).sum()):5} valid gridcells")
print(f"common: {int(area.sum()):5} of {int((mask == 1).sum())} land gridcells")

fig = plotting.plot_mask_agreement(
    valid, mask == 1, title=f"Area masks of {', '.join(inputs)} ({GRID_TAG})",
    label="datasets with the gridcell valid", highlight_common=True,
)
save_fig(fig, f"all.area_mask.{GRID_TAG}.png")

In [ ]:
if SAVE:
    fout = PROC_OUT / f"all.area_mask.{GRID_TAG}.nc"
    fout.parent.mkdir(parents=True, exist_ok=True)
    area.astype("i1").assign_attrs(
        long_name="land gridcells where ET (any year), LAI and AI are valid in every member of every dataset",
        datasets=list(inputs),
        time_periods=[f"{name}: {ds.attrs['time_period']}" for name, ds in inputs.items()],
        members=[f"{name}.{m}" for name, ds in inputs.items() for m in ds.member.values],
    ).to_netcdf(fout)
    print(fout)

# Binning inputs restricted to the common area
common = {name: ds[["lai", "ai"]].where(area) for name, ds in inputs.items()}

## 3. Hypothesis: CMIP6 spans the widest range of LAI and AI

Two readings of "range": (a) the range of the values a dataset covers (min to max of all its members pooled), and (b) the spread of the edges across the members of a dataset (models for CMIP6). Both are computed on each dataset's own area and on the common area. Quantile edges 0 and N are the pooled min and max, so (a) is also the range of each dataset's own pooled edges.

In [ ]:
rows = []
for v in N_BINS:
    for name, ds in inputs.items():
        for area_label, a in (("own", ds.valid), ("common", area)):
            x = binning.finite_flat(ds[v].where(a))
            p1, p99 = np.percentile(x, [1, 99])
            rows.append({
                "variable": v, "area": area_label, "dataset": name, "n_values": x.size,
                "min": x.min(), "p1": p1, "p99": p99, "max": x.max(),
                "range": x.max() - x.min(), "range_p1_p99": p99 - p1,
            })
ranges = pd.DataFrame(rows).set_index(["variable", "area", "dataset"])
with pd.option_context("display.float_format", "{:0.4g}".format):
    display(ranges)

In [ ]:
# (a) Is CMIP6's range the widest, and does it contain the range of every other dataset?
for (v, area_label), r in ranges.groupby(level=["variable", "area"]):
    r = r.droplevel(["variable", "area"])
    others = r.drop(index="cmip")
    print(
        f"{v:3} {area_label:6}: widest range {r['range'].idxmax():4} (p1-p99: {r['range_p1_p99'].idxmax():4}); "
        f"CMIP6 [{r.loc['cmip', 'min']:0.3g}, {r.loc['cmip', 'max']:0.3g}] contains the others: "
        f"min {bool((others['min'] >= r.loc['cmip', 'min']).all())}, max {bool((others['max'] <= r.loc['cmip', 'max']).all())}"
    )

In [ ]:
def member_edges(ds: xr.Dataset, v: str) -> xr.DataArray:
    """Quantile edges (member, edge) of each member of `ds` on its own."""
    return concat_members([
        binning.pooled_bin_edges(ds[v].sel(member=m), N_BINS[v], name=v, verbose=False) for m in ds.member.values
    ]).assign_coords(member=ds.member.values)


# (b) Spread of the edges across the members of each dataset, on the common area
mem_edges = {v: {name: member_edges(ds, v) for name, ds in common.items()} for v in N_BINS}
rows = []
for v, by_name in mem_edges.items():
    for name, e in by_name.items():
        spread = (e.max("member") - e.min("member")).values
        rows.append({
            "variable": v, "dataset": name, "n_members": e.sizes["member"],
            "spread_min_edge": spread[0], "spread_max_edge": spread[-1],
            "mean_spread_interior_edges": spread[1:-1].mean(), "max_spread_interior_edges": spread[1:-1].max(),
        })
spreads = pd.DataFrame(rows).set_index(["variable", "dataset"])
with pd.option_context("display.float_format", "{:0.4g}".format):
    display(spreads)
for v, r in spreads.groupby(level="variable"):
    r = r.droplevel("variable")
    print(f"{v:3}: largest mean spread of the interior edges across members: {r['mean_spread_interior_edges'].idxmax()}")

In [ ]:
# Envelope (min to max across members) of the per-member edges, without the maximum edge
fig, axs = plt.subplots(1, 2, figsize=(12, 4.5), layout="constrained")
for ax, (v, by_name) in zip(axs, mem_edges.items()):
    for name, e in by_name.items():
        e = e.isel(edge=slice(None, -1))
        ax.fill_between(e.edge, e.min("member"), e.max("member"), color=COLORS[name], alpha=0.2)
        ax.plot(e.edge, e.median("member"), color=COLORS[name], label=f"{LABELS[name]} ({e.sizes['member']})")
    ax.set_xlabel("edge")
    ax.set_ylabel(f"{v} [{binning.EDGE_ATTRS[v]['units']}]")
    ax.set_title(f"{v}: per-member edges, median and min-max (common area)")
    ax.legend()
save_fig(fig, f"all.lai_ai_clim.quantiles_member_spread.{GRID_TAG}.png")

## 4. Edges pooled across all datasets vs each dataset's own pooled edges

Each dataset's own edges pool its members as in its script (`binning.pooled_bin_edges`), here on the common area. The edges pooled across all datasets either weigh each dataset equally (`"dataset"`, weighted quantiles) or count every finite value once (`"value"`, `binning.pooled_bin_edges`). On the common area every member of a dataset has the same number of values, so with `"dataset"` each member weighs 1 / (4 * n_members of its dataset).

In [ ]:
def dataset_weighted_edges(fields: dict[str, xr.DataArray], n_bins: int, name: str, attrs: dict) -> xr.DataArray:
    """
    Quantile edges of all `fields` pooled, each field weighing equally whatever its size.
    Same format as binning.pooled_bin_edges, which counts every value once.
    """
    values = [binning.finite_flat(da) for da in fields.values()]
    weights = np.concatenate([np.full(x.size, 1 / x.size) for x in values])
    # Weighted quantiles need method="inverted_cdf" (edges fall on data values, not between
    # them as in build_edges); edges 0 and n_bins are the pooled min and max
    edges = np.quantile(np.concatenate(values), np.linspace(0, 1, n_bins + 1), weights=weights, method="inverted_cdf")
    template = binning.pooled_bin_edges(fields, n_bins, name=name, verbose=False, attrs=attrs)
    return template.copy(data=edges)


period_attrs = {"time_periods": [f"{name}: {ds.attrs['time_period']}" for name, ds in inputs.items()], "grid": GRID_TAG}
own_edges = {v: {} for v in N_BINS}
all_edges = {v: {} for v in N_BINS}
for v in N_BINS:
    attrs = {**binning.EDGE_ATTRS[v], **period_attrs}
    for name, ds in common.items():
        own_edges[v][name] = binning.pooled_bin_edges(ds[v], N_BINS[v], name=v, verbose=False, attrs=attrs)
    all_edges[v]["value"] = binning.pooled_bin_edges(
        {name: ds[v] for name, ds in common.items()}, N_BINS[v], name=v, verbose=False,
        attrs={**attrs, "weighting": "value"},
    )
    all_edges[v]["dataset"] = dataset_weighted_edges(
        {name: ds[v] for name, ds in common.items()}, N_BINS[v], v, attrs={**attrs, "weighting": "dataset"},
    )

edge_table = pd.concat(
    {v: pd.DataFrame({**{LABELS[n]: e.values for n, e in own_edges[v].items()},
                      **{f"all ({w})": e.values for w, e in all_edges[v].items()}})
     for v in N_BINS},
    names=["variable", "edge"],
)
with pd.option_context("display.float_format", "{:0.4g}".format, "display.max_rows", 40):
    display(edge_table)

In [ ]:
# Edges of each dataset and pooled across all datasets; right: without the maximum edge
fig, axs = plt.subplots(2, 2, figsize=(12, 8), layout="constrained")
for row, v in zip(axs, N_BINS):
    for ax, drop_last in zip(row, (False, True)):
        sl = slice(None, -1) if drop_last else slice(None)
        for name, e in own_edges[v].items():
            ax.plot(e.edge[sl], e[sl], color=COLORS[name], marker="o", ms=3, lw=1, alpha=0.8, label=LABELS[name])
        for w, ls in (("dataset", "-"), ("value", "--")):
            e = all_edges[v][w]
            ax.plot(e.edge[sl], e[sl], color="m", ls=ls, lw=2, label=f"all, {w} weighting")
        ax.set_xlabel("edge")
        ax.set_ylabel(f"{v} [{binning.EDGE_ATTRS[v]['units']}]")
        ax.set_title(f"{v}" + (", without the maximum edge" if drop_last else ""))
        if v == "ai" and not drop_last:
            ax.set_yscale("symlog", linthresh=1)
axs[0, 0].legend()
fig.suptitle(f"Quantile edges on the common area ({int(area.sum())} gridcells)")
save_fig(fig, f"all.lai_ai_clim.quantiles_pooled_vs_own.{GRID_TAG}.png")

How well do the shared edges resolve each dataset? With its own quantile edges, every bin holds 1/N of a dataset's values. With the shared edges, the fraction per bin shows where a dataset is squeezed into few bins or leaves bins nearly empty. Values are assigned to bins as in `binning.bin_stats`.

In [ ]:
def bin_index(x: np.ndarray, edges: np.ndarray) -> np.ndarray:
    """Bin of each value, as in binning._bin_stats_flat: [lower, upper), out-of-range values clipped into the edge bins."""
    return np.clip(np.searchsorted(edges, x, side="right") - 1, 0, len(edges) - 2)


def bin_fraction(x: np.ndarray, edges: np.ndarray) -> np.ndarray:
    return np.bincount(bin_index(x, edges), minlength=len(edges) - 1) / x.size


fig, axs = plt.subplots(1, 2, figsize=(12, 3.5), layout="constrained")
for ax, v in zip(axs, N_BINS):
    edges = all_edges[v][WEIGHTING].values
    frac = np.stack([bin_fraction(binning.finite_flat(ds[v]), edges) for ds in common.values()])
    pm = ax.pcolormesh(np.arange(N_BINS[v] + 1), np.arange(len(common) + 1), frac,
                       cmap="RdBu_r", vmin=0, vmax=2 / N_BINS[v])
    for (i, j), f in np.ndenumerate(frac):
        ax.text(j + 0.5, i + 0.5, f"{100 * f:.0f}", ha="center", va="center", fontsize=7)
    ax.set_yticks(np.arange(len(common)) + 0.5, [LABELS[n] for n in common])
    ax.set_xlabel(f"{v} bin")
    ax.set_title(f"{v}: % of values per bin, shared edges ({WEIGHTING} weighting); even = {100 / N_BINS[v]:.1f}%")
    fig.colorbar(pm, ax=ax, label="fraction")
save_fig(fig, f"all.lai_ai_clim.quantiles_pooled_{WEIGHTING}.bin_fraction.{GRID_TAG}.png")

In [ ]:
# Gridcells per 2-D (lai, ai) bin with the shared edges, mean over each dataset's members
y_edges, x_edges = all_edges["lai"][WEIGHTING].values, all_edges["ai"][WEIGHTING].values
counts = {}
for name, ds in common.items():
    lai, ai = (ds[v].values.ravel() for v in ("lai", "ai"))
    ok = np.isfinite(lai) & np.isfinite(ai)
    lin = bin_index(lai[ok], y_edges) * N_BINS["ai"] + bin_index(ai[ok], x_edges)
    counts[name] = np.bincount(lin, minlength=N_BINS["lai"] * N_BINS["ai"]).reshape(N_BINS["lai"], N_BINS["ai"]) / ds.sizes["member"]

fig, axs = plt.subplots(1, len(counts), figsize=(4.2 * len(counts), 4), layout="constrained", sharey=True)
vmax = max(np.percentile(c, 98) for c in counts.values())
for ax, (name, c) in zip(axs, counts.items()):
    pm = ax.pcolormesh(np.ma.masked_equal(c, 0), cmap="viridis", vmin=0, vmax=vmax)
    ax.set_title(f"{LABELS[name]}: {int((c == 0).sum())} empty bins")
    ax.set_xlabel("ai bin")
axs[0].set_ylabel("lai bin")
fig.colorbar(pm, ax=axs, label="gridcells per member (white: none)", extend="max")
fig.suptitle(f"Gridcells per bin, shared edges ({WEIGHTING} weighting)")
save_fig(fig, f"all.lai_ai_clim.quantiles_pooled_{WEIGHTING}.bin_count.{GRID_TAG}.png")

n_min = cfg.SIGNIF_N_MIN
shared = np.logical_and.reduce([c * inputs[n].sizes["member"] > n_min for n, c in counts.items()])
print(f"{int(shared.sum())} of {shared.size} bins have > {n_min} gridcell samples in every dataset")

## 5. Save the pooled edges

The edges pooled across all datasets with `WEIGHTING`, for the scripts to share.

In [ ]:
if SAVE:
    for v in N_BINS:
        e = all_edges[v][WEIGHTING].assign_attrs(
            datasets=list(inputs),
            members=[f"{name}.{m}" for name, ds in inputs.items() for m in ds.member.values],
            area_mask=str(PROC_OUT / f"all.area_mask.{GRID_TAG}.nc"),
        )
        fout = EDGES_OUT / f"all.{v}_clim.{N_BINS[v]}_quantiles_pooled_{WEIGHTING}.{GRID_TAG}.nc"
        fout.parent.mkdir(parents=True, exist_ok=True)
        e.to_netcdf(fout)
        print(fout)